In [5]:
!pip install nltk pyxDamerauLevenshtein

%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import optuna 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader 
from sklearn.model_selection import train_test_split
from DataEncoder import encode_pad_event, encode_pad_sequence, scale_time_differences, encode_label_event, node_time_list, scale_time_differences_fast_fixed, length_stratified_split
from GATConv import prepare_data_core_timedif, prepare_data_y, CustomDataset, EarlyStopping, train, evaluate, custom_collate_fn, DualGATModel
from GATConv import predict, top_k_accuracy, predict_per_sequence, average_bleu_score, compute_dls_and_exact_match, sequence_level_top_k_accuracy, analyze_sequence_errors, predict_per_sequence_with_probs, sequence_level_top_k_analysis, show_error_sequences
import os 
import shutil

In [6]:
#event = pd.read_csv("../output/BPI12.csv")
# event = pd.read_csv("../output/BPI12w.csv")
#event = pd.read_csv("../output/BPI13i.csv")
#event = pd.read_csv("../output/BPI13c.csv")
event = pd.read_csv("../output/helpdesk.csv")

In [7]:
# check the size of sequence
shortest_sequence = event.groupby('sequence').size().min()
longest_sequence = event.groupby('sequence').size().max()
print('shortest_sequence:', shortest_sequence)
print('longest_sequence:', longest_sequence)
event = event[event.groupby('sequence')['sequence'].transform('size') >= 2].reset_index(drop=True)

shortest_sequence: 2
longest_sequence: 15


In [8]:
#core_event = 'event_label' 
core_event = 'event' # with status
case_index = 'sequence'

In [9]:
core_encode, y_encode, core_size, output_size, le_event = encode_label_event(event, core_event, case_index)

In [10]:
#BPI12 
#event['ec1'] = event['ec1'].astype(str) 
# cat_col_event = ['ec1']
#BPI17
#cat_col_event = ['ec1', 'ec2', 'ec3']
#BPI13
#cat_col_event = ['ec1', 'ec2', 'ec4', 'ec5']
#helpdesk
cat_col_event = ['ec1']
num_col_event = []
event_encode = encode_pad_event(event, cat_col_event, num_col_event, case_index, cat_mask = True, num_mask = True, eos = False)

In [11]:
#BPI12
# sequence = event[['sequence','sn1']].groupby('sequence').first()
#BPI17
#sequence = event[['sequence','sn1', 'sc1', 'sc2']].groupby('sequence').first()
#BPI13
#sequence = event[['sequence','sc3', 'sc1', 'sc2']].groupby('sequence').first()
#helpdesk
sequence = event[['sequence','sc1']].groupby('sequence').first()
sequence = sequence.reset_index()
#sequence = event.groupby('sequence').apply(lambda x: x.iloc[prefix_size - 1:]).reset_index(drop=True)

In [12]:
#bpi12
# cat_col_seq = []
# num_col_seq = ['sn1']
#Bpi13
#cat_col_seq = ['sc1','sc2','sc3']
#num_col_seq = []
#helpdesk
cat_col_seq = ['sc1']
num_col_seq = []
sequence_encode = encode_pad_sequence(sequence, cat_col_seq, num_col_seq)

In [13]:
start_time_col = 'time'
#start_time_col = 'Start Timestamp'
scaled_time_diffs = scale_time_differences_fast_fixed(event, sequence, start_time_col, case_index)

In [14]:
max_num_events = event_encode.shape[1]
# Expand sequence features to match the shape of event features
sequence_features_expanded = np.expand_dims(sequence_encode, axis=1)
sequence_features_expanded = np.repeat(sequence_features_expanded, max_num_events, axis=1)

# Combine event and sequence features
combined_features = np.concatenate((event_encode, sequence_features_expanded), axis=2)

In [15]:
num_sequences = event_encode.shape[0]
num_event_features = combined_features.shape[2]
num_embedding_features = core_size

In [16]:
node_times = node_time_list(event, start_time_col, case_index)

In [17]:
# graph data preparation
event_feature_list = prepare_data_core_timedif(combined_features, core_encode, scaled_time_diffs, node_times)
y_list = prepare_data_y(combined_features, y_encode)

In [18]:
# Use stratified sampling over sequence length to preserve distributional characteristics
train_indices, test_indices = length_stratified_split(event_feature_list, test_size=0.2, n_bins=10)

# Split the data
train_event_features = [event_feature_list[i] for i in train_indices]
test_event_features = [event_feature_list[i] for i in test_indices]
train_y = [y_list[i] for i in train_indices]
test_y = [y_list[i] for i in test_indices]

# Print statistics
train_lengths = [event_feature_list[i].x.shape[0] for i in train_indices]
test_lengths = [event_feature_list[i].x.shape[0] for i in test_indices]

print(f"Train set: {len(train_indices)} samples")
print(f"Train length range: {min(train_lengths)} - {max(train_lengths)}")
print(f"Train length mean: {np.mean(train_lengths):.2f}")

print(f"\nTest set: {len(test_indices)} samples") 
print(f"Test length range: {min(test_lengths)} - {max(test_lengths)}")
print(f"Test length mean: {np.mean(test_lengths):.2f}")

Train set: 3666 samples
Train length range: 2 - 14
Train length mean: 4.57

Test set: 914 samples
Test length range: 3 - 15
Test length mean: 4.88


In [19]:
train_dataset = CustomDataset(train_event_features, train_y)
test_dataset = CustomDataset(test_event_features, test_y)

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# parameters
embedding_dims = 64
gat_hidden_dim_event = 32
gat_hidden_dim_embed = 128
gat_hidden_dim_concat = 256
output_dim = output_size  # vocab size
num_heads = 4

model = DualGATModel(
    num_event_features=num_event_features,
    num_embedding_features=num_embedding_features,
    embedding_dims=embedding_dims,
    gat_hidden_dim_event=gat_hidden_dim_event,
    gat_hidden_dim_embed=gat_hidden_dim_embed,
    gat_hidden_dim_concat=gat_hidden_dim_concat,
    output_dim=output_dim,
    num_heads=num_heads
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=-1)  # ignore padding

In [21]:
batch_size = 16  
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

In [23]:
os.makedirs(os.path.dirname(model_path), exist_ok=True)

In [24]:
early_stopping = EarlyStopping(patience=3, delta=0.0)
num_epochs = 10

# Filepath to save model
model_path = "../output/models/BPI12_timeedge_es.pt"

config = {
    'model_type': 'DualGATModel',
    'num_event_features': num_event_features,
    'num_embedding_features': num_embedding_features,
    'embedding_dims': embedding_dims,
    'gat_hidden_dim_event': gat_hidden_dim_event,
    'gat_hidden_dim_embed': gat_hidden_dim_embed,
    'gat_hidden_dim_concat': gat_hidden_dim_concat,
    'output_dim': output_dim,
    'num_heads': num_heads
}

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    if early_stopping(test_loss):
        print("Early stopping triggered.")
        break

    if early_stopping.best_loss_updated:
        print(f"New best model at epoch {epoch+1}, saving to {model_path}")
        best_model_saved = True
         # Save state dict and config
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)
        
if not early_stopping.early_stop and not best_model_saved:
        print("Training completed without early stopping. Saving final model.")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)

Epoch 1/10 | Train Loss: 0.2185 Acc: 0.9251 | Test Loss: 0.2985 Acc: 0.9072
New best model at epoch 1, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 2/10 | Train Loss: 0.1754 Acc: 0.9354 | Test Loss: 0.2893 Acc: 0.9195
New best model at epoch 2, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 3/10 | Train Loss: 0.1584 Acc: 0.9394 | Test Loss: 0.2866 Acc: 0.9193
New best model at epoch 3, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 4/10 | Train Loss: 0.1401 Acc: 0.9448 | Test Loss: 0.2931 Acc: 0.9166
Epoch 5/10 | Train Loss: 0.1279 Acc: 0.9475 | Test Loss: 0.2732 Acc: 0.9244
New best model at epoch 5, saving to ../output/models/BPI12_timeedge_es.pt
Epoch 6/10 | Train Loss: 0.1220 Acc: 0.9489 | Test Loss: 0.2831 Acc: 0.9229
Epoch 7/10 | Train Loss: 0.1216 Acc: 0.9491 | Test Loss: 0.2831 Acc: 0.9190
Epoch 8/10 | Train Loss: 0.1120 Acc: 0.9529 | Test Loss: 0.2839 Acc: 0.9260
Early stopping triggered.


In [25]:
preds, labels, outs = predict(model, test_loader, device)

# Check some predictions
for i in range(10):
    print(f"Predicted: {preds[i].item()} | Actual: {labels[i].item()}")

Predicted: 10 | Actual: 10
Predicted: 1 | Actual: 1
Predicted: 5 | Actual: 5
Predicted: 10 | Actual: 10
Predicted: 1 | Actual: 1
Predicted: 5 | Actual: 5
Predicted: 10 | Actual: 10
Predicted: 1 | Actual: 1
Predicted: 5 | Actual: 5
Predicted: 10 | Actual: 10


In [26]:
top1 = top_k_accuracy(outs, labels, k=1)
top3 = top_k_accuracy(outs, labels, k=3)
top5 = top_k_accuracy(outs, labels, k=5)

print(f"Top-1 Acc: {top1:.4f} | Top-3 Acc: {top3:.4f} | Top-5 Acc: {top5:.4f}")

Top-1 Acc: 0.9260 | Top-3 Acc: 0.9865 | Top-5 Acc: 0.9957


In [27]:
preds_seq, labels_seq = predict_per_sequence(model, test_loader, device)

In [28]:
bleu = average_bleu_score(preds_seq, labels_seq)
print(f"Average BLEU Score: {bleu:.4f}")

Average BLEU Score: 0.8657


In [29]:
avg_dls, exact_acc = compute_dls_and_exact_match(preds_seq, labels_seq)
print(f"Damerau-Levenshtein Similarity (avg): {avg_dls:.4f}")
print(f"Exact Match Accuracy: {exact_acc:.4f}")


Damerau-Levenshtein Similarity (avg): 0.9456
Exact Match Accuracy: 0.7932


In [30]:
# Usage
seq_top3_acc = sequence_level_top_k_accuracy(model, test_loader, device, k=3)
print(f"Sequence-Level Top-3 Accuracy: {seq_top3_acc:.4f}")

seq_top5_acc = sequence_level_top_k_accuracy(model, test_loader, device, k=5)
print(f"Sequence-Level Top-5 Accuracy: {seq_top5_acc:.4f}")

Sequence-Level Top-3 Accuracy: 0.9606
Sequence-Level Top-5 Accuracy: 0.9836


In [31]:
# Usage example
pos_errors, error_types, seq_stats = analyze_sequence_errors(model, test_loader, device, k=3)


=== Error Analysis Report ===
Total sequences analyzed: 914
Total sequences with errors: 189
Sequence length stats: {'min': 3, 'max': 15, 'mean': 4.878555798687089, 'median': 4.0}

Error distribution by position:
Position 0: 34 errors (10.3% of all errors)
Position 1: 59 errors (17.9% of all errors)
Position 2: 89 errors (27.0% of all errors)
Position 3: 41 errors (12.4% of all errors)
Position 4: 62 errors (18.8% of all errors)
Position 5: 18 errors (5.5% of all errors)
Position 6: 13 errors (3.9% of all errors)
Position 7: 4 errors (1.2% of all errors)
Position 8: 3 errors (0.9% of all errors)
Position 9: 3 errors (0.9% of all errors)
Position 10: 2 errors (0.6% of all errors)
Position 11: 2 errors (0.6% of all errors)

Most common error types:
Predicted 10 instead of 1: 46 times
Predicted 10 instead of 14: 40 times
Predicted 14 instead of 10: 32 times
Predicted 12 instead of 0: 30 times
Predicted 10 instead of 12: 25 times
Predicted 0 instead of 12: 22 times
Predicted 12 instead of

In [32]:
# First get predictions with probabilities
all_preds, all_labels, all_topk = predict_per_sequence_with_probs(model, test_loader, device, k=3)

# Then analyze sequence-level top-k accuracy
topk_acc, error_stats = sequence_level_top_k_analysis(all_topk, all_labels)

print(f"Sequence-Level Top-3 Accuracy: {topk_acc:.4f}")
print("\nError Analysis:")
print(f"- Most error-prone positions: {error_stats['position_errors']}")
print(f"- Top prediction mistakes: {error_stats['top_errors']}")

Sequence-Level Top-3 Accuracy: 0.9606

Error Analysis:
- Most error-prone positions: {0: 1, 1: 13, 2: 16, 3: 9, 4: 9, 5: 4, 6: 4, 7: 2, 8: 0, 9: 2}
- Top prediction mistakes: {(14, 2): 8, (10, 12): 8, (10, 14): 5, (1, 12): 5, (10, 8): 4}


In [33]:
show_error_sequences(all_topk, all_labels, num=3)


Error Sequence #556:
True: [0, 0, 10, 10, 1, 5]
Pred: [12, 12, 12, 12, 1, 5]
Mismatches:
Pos 2: True 10 not in [12, 0, 14]
Pos 3: True 10 not in [12, 0, 14]

Error Sequence #566:
True: [12, 2, 8, 8, 10, 5]
Pred: [12, 14, 10, 10, 10, 1]
Mismatches:
Pos 1: True 2 not in [14, 10, 8]
Pos 3: True 8 not in [10, 14, 5]

Error Sequence #567:
True: [12, 10, 14, 10, 1, 5]
Pred: [12, 14, 10, 1, 1, 5]
Mismatches:
Pos 2: True 14 not in [10, 1, 12]


In [34]:
from sklearn.metrics import classification_report
unique_true = np.unique(labels.cpu().numpy())
unique_preds = np.unique(preds.cpu().numpy())

print("Classes in true labels:", unique_true)
print("Classes in predictions:", unique_preds)
print("Missing classes:", set(unique_true) - set(unique_preds))

actual_classes = np.union1d(unique_true, unique_preds)
report = classification_report(
    labels.cpu().numpy(),
    preds.cpu().numpy(),
    target_names=[f"Class_{i}" for i in actual_classes] ,
    digits=4
)
print(report)

Classes in true labels: [ 0  1  2  5  8  9 10 11 12 14]
Classes in predictions: [ 0  1  2  5  8  9 10 12 14]
Missing classes: {11}
              precision    recall  f1-score   support

     Class_0     0.6625    0.6386    0.6503        83
     Class_1     0.9573    0.9394    0.9482       907
     Class_2     0.6667    0.1250    0.2105        16
     Class_5     0.9967    0.9945    0.9956       914
     Class_8     0.6957    0.4848    0.5714        33
     Class_9     0.0000    0.0000    0.0000         4
    Class_10     0.8798    0.9361    0.9071      1001
    Class_11     0.0000    0.0000    0.0000         2
    Class_12     0.9444    0.9346    0.9395      1146
    Class_14     0.8234    0.8187    0.8210       353

    accuracy                         0.9260      4459
   macro avg     0.6626    0.5872    0.6044      4459
weighted avg     0.9243    0.9260    0.9241      4459



/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [35]:
# Load model state
checkpoint = torch.load(model_path, map_location=device)
config = checkpoint['config']

# Rebuild the model using the same config
model = DualGATModel(
    num_event_features=config['num_event_features'],
    num_embedding_features=config['num_embedding_features'],
    embedding_dims=config['embedding_dims'],
    gat_hidden_dim_event=config['gat_hidden_dim_event'],
    gat_hidden_dim_embed=config['gat_hidden_dim_embed'],
    gat_hidden_dim_concat=config['gat_hidden_dim_concat'],
    output_dim=config['output_dim'],
    num_heads=config['num_heads']
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

# Rebuild optimizer (if needed)
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Recover attention and event IDs
start_epoch = checkpoint['epoch'] + 1 

for epoch in range(start_epoch, 10):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    if early_stopping(test_loss):
        print("Early stopping triggered.")
        break

    if early_stopping.best_loss_updated:
        print(f"New best model at epoch {epoch+1}, saving to {model_path}")
        best_model_saved = True
         # Save state dict and config
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)
        
if not early_stopping.early_stop and not best_model_saved:
        print("Training completed without early stopping. Saving final model.")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)

Epoch 6/10 | Train Loss: 0.1265 Acc: 0.9481 | Test Loss: 0.2739 Acc: 0.9296
Early stopping triggered.
